# S11-19 — Feature Importance Pronostia — Scénario by_condition

| Champ | Valeur |
|-------|--------|
| **Expériences** | exp_102 (KMeans), exp_105 (Mahalanobis), exp_108 (EWC), exp_111 (HDC) |
| **Scénario** | by_condition : condition_1 → condition_2 → condition_3 |
| **Features** | 13 features : 6×acc_horiz + 6×acc_vert + temporal_position |
| **Sprint** | Sprint 11 — S11-19 |


## Section 1 — Setup, imports et chargement des résultats

In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path('.').resolve()
if _cwd.name == 'pronostia_feature_importance':
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == 'cl_eval':
    os.chdir(_cwd.parent.parent)

sys.path.insert(0, str(Path('.').resolve()))

from src.evaluation.feature_importance import (
    CHANNEL_GROUPS_PRONOSTIA,
    FEATURE_NAMES_PRONOSTIA,
    plot_feature_importance,
    plot_feature_importance_comparison,
)

FIGURES_DIR = Path("notebooks/figures/cl_evaluation/pronostia_feature_importance")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXPS = {
    "KMeans":      "experiments/exp_102/results/feature_importance.json",
    "Mahalanobis": "experiments/exp_105/results/feature_importance.json",
    "EWC":         "experiments/exp_108/results/feature_importance.json",
    "HDC":         "experiments/exp_111/results/feature_importance.json",
}

data = {model: json.load(open(path)) for model, path in EXPS.items()}
TASKS = ["condition_1", "condition_2", "condition_3"]
print("Chargement OK — modèles :", list(data.keys()))
print("Features :", FEATURE_NAMES_PRONOSTIA)
print("Groupes canaux :", {g: len(f) for g, f in CHANNEL_GROUPS_PRONOSTIA.items()})


Chargement OK — modèles : ['KMeans', 'Mahalanobis', 'EWC', 'HDC']
Features : ['mean_acc_horiz', 'std_acc_horiz', 'rms_acc_horiz', 'kurtosis_acc_horiz', 'peak_acc_horiz', 'crest_factor_acc_horiz', 'mean_acc_vert', 'std_acc_vert', 'rms_acc_vert', 'kurtosis_acc_vert', 'peak_acc_vert', 'crest_factor_acc_vert', 'temporal_position']
Groupes canaux : {'acc_horiz': 6, 'acc_vert': 6, 'temporal': 1}


## Section 2 — Tableau récapitulatif global (ranking par modèle)

In [2]:
rows = {}
for model, d in data.items():
    scores = d["global"]["permutation_importance"]
    rows[model] = {f: scores.get(f, 0.0) for f in FEATURE_NAMES_PRONOSTIA}

df = pd.DataFrame(rows).T
df_rank = df.rank(axis=1, ascending=False).astype(int)

print("=== Scores d'importance (permutation globale) ===")
display(df.round(4))
print()
print("=== Rang par feature (1 = plus importante) ===")
display(df_rank)


=== Scores d'importance (permutation globale) ===


,mean_acc_horiz,std_acc_horiz,rms_acc_horiz,kurtosis_acc_horiz,peak_acc_horiz,crest_factor_acc_horiz,mean_acc_vert,std_acc_vert,rms_acc_vert,kurtosis_acc_vert,peak_acc_vert,crest_factor_acc_vert,temporal_position
KMeans,-0.0007,0.0078,0.0088,0.0070,0.0072,0.0101,-0.0025,0.0017,0.0033,0.0137,0.0101,0.0038,0.0037
Mahalanobis,-0.0004,0.7021,0.6996,0.0195,0.0690,0.0467,0.0013,0.7292,0.7271,0.0442,0.0703,0.0499,0.0012
EWC,-0.0008,0.0088,0.0068,0.0036,0.0012,0.0107,0.0033,0.0176,0.0321,0.0054,0.0008,0.0077,0.1247
HDC,-0.0009,-0.0168,0.0021,0.0008,0.0000,0.0107,-0.0007,-0.0436,-0.0410,0.0029,-0.0237,0.0008,0.0062



=== Rang par feature (1 = plus importante) ===


,mean_acc_horiz,std_acc_horiz,rms_acc_horiz,kurtosis_acc_horiz,peak_acc_horiz,crest_factor_acc_horiz,mean_acc_vert,std_acc_vert,rms_acc_vert,kurtosis_acc_vert,peak_acc_vert,crest_factor_acc_vert,temporal_position
KMeans,12,5,4,7,6,2,13,11,10,1,3,8,9
Mahalanobis,13,3,4,10,6,8,11,1,2,9,5,7,12
EWC,13,5,7,9,11,4,10,3,2,8,12,6,1
HDC,9,10,4,5,7,1,8,13,12,3,11,5,2


## Section 3 — Barplot groupé global (4 modèles × 13 features)

In [3]:
importances_dict = {
    model: d["global"]["permutation_importance"]
    for model, d in data.items()
}

fig = plot_feature_importance_comparison(
    importances_dict,
    FEATURE_NAMES_PRONOSTIA,
    title="Importance globale — Pronostia by_condition (4 modèles)",
)
fig.savefig(FIGURES_DIR / "by_condition_global_comparison.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_condition_global_comparison.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/pronostia_feature_importance/by_condition_global_comparison.png


/tmp/ipykernel_95744/3309650551.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 4 — Barplot par canal (acc_horiz vs acc_vert vs temporal)

In [4]:
# Aggrégation par canal : somme des importances dans chaque groupe
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Importance par canal — Pronostia by_condition", fontsize=13, fontweight="bold")
colors = {"acc_horiz": "#2196F3", "acc_vert": "#4CAF50", "temporal": "#FF9800"}

for ax, (model, d) in zip(axes, data.items()):
    scores = d["global"]["permutation_importance"]
    channel_scores = {
        ch: sum(scores.get(f, 0.0) for f in feats)
        for ch, feats in CHANNEL_GROUPS_PRONOSTIA.items()
    }
    ch_names = list(channel_scores.keys())
    ch_vals = list(channel_scores.values())
    bar_colors = [colors[c] for c in ch_names]
    ax.bar(ch_names, ch_vals, color=bar_colors, edgecolor="white", alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(model, fontsize=11, fontweight="bold")
    ax.set_ylabel("Importance agrégée", fontsize=9)
    ax.set_xlabel("Canal", fontsize=9)
    for i, v in enumerate(ch_vals):
        ax.text(i, v + max(abs(v) * 0.03, 0.001), f"{v:+.4f}", ha="center", fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "by_condition_channel_grouped.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_condition_channel_grouped.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/pronostia_feature_importance/by_condition_channel_grouped.png


/tmp/ipykernel_95744/3393936327.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5 — Heatmap per-task (3 conditions × 13 features) par modèle

In [5]:
fig, axes = plt.subplots(1, 4, figsize=(26, 5))
fig.suptitle("Importance par condition opératoire — Pronostia by_condition", fontsize=13, fontweight="bold")

for ax, (model, d) in zip(axes, data.items()):
    mat = np.array([
        [d["per_task"][t]["permutation_importance"].get(f, 0.0) for f in FEATURE_NAMES_PRONOSTIA]
        for t in TASKS
    ])
    vmax = max(abs(mat).max(), 1e-6)
    im = ax.imshow(mat, aspect="auto", cmap="RdYlGn", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(FEATURE_NAMES_PRONOSTIA)))
    ax.set_xticklabels(FEATURE_NAMES_PRONOSTIA, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(TASKS)))
    ax.set_yticklabels(TASKS, fontsize=9)
    ax.set_title(model, fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "by_condition_per_task_heatmap.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_condition_per_task_heatmap.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/pronostia_feature_importance/by_condition_per_task_heatmap.png


/tmp/ipykernel_95744/3798806101.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6 — Focus `temporal_position` : importance à travers les 4 modèles

In [6]:
# FIXME(gap1) : temporal_position peut ne pas être disponible en déploiement MCU
temporal_scores = {
    model: d["global"]["permutation_importance"].get("temporal_position", 0.0)
    for model, d in data.items()
}

fig, ax = plt.subplots(figsize=(7, 4))
models = list(temporal_scores.keys())
scores = list(temporal_scores.values())
bar_colors = ["#4CAF50" if s >= 0 else "#F44336" for s in scores]
ax.bar(models, scores, color=bar_colors, edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Importance de temporal_position — 4 modèles\n(FIXME(gap1): non disponible MCU sans horodatage)", fontsize=11)
ax.set_ylabel("Importance (permutation)", fontsize=10)
for i, (m, v) in enumerate(zip(models, scores)):
    ax.text(i, v + max(abs(v) * 0.03, 0.001), f"{v:+.4f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "temporal_position_importance.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'temporal_position_importance.png'}")

temporal_max = max(abs(v) for v in temporal_scores.values())
print()
if temporal_max > 0.01:
    print("⚠️  FIXME(gap1) : temporal_position est significative (max abs =", f"{temporal_max:.4f})")
    print("   → À retirer pour les expériences de publication (non dispo MCU sans horodatage)")
else:
    print("✅ temporal_position non significative — retrait sans impact sur les performances")


Figure sauvegardée : notebooks/figures/cl_evaluation/pronostia_feature_importance/temporal_position_importance.png

⚠️  FIXME(gap1) : temporal_position est significative (max abs = 0.1247)
   → À retirer pour les expériences de publication (non dispo MCU sans horodatage)


/tmp/ipykernel_95744/3916820472.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7 — EWC et HDC : comparaison des méthodes d'importance

In [7]:
def minmax(d):
    vals = np.array(list(d.values()))
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.0 for k in d}
    return {k: (v - mn) / (mx - mn) for k, v in d.items()}

# EWC : permutation vs gradient saliency
ewc = data["EWC"]["global"]
perm_ewc = minmax(ewc["permutation_importance"])
grad_ewc  = minmax(ewc["gradient_saliency"])

fig = plot_feature_importance_comparison(
    {"Permutation (EWC)": perm_ewc, "Gradient Saliency (EWC)": grad_ewc},
    FEATURE_NAMES_PRONOSTIA,
    title="EWC — Permutation vs Gradient Saliency (normalisés) — Pronostia by_condition",
)
plt.show()

from scipy.stats import spearmanr
rho_ewc, _ = spearmanr(
    [perm_ewc[f] for f in FEATURE_NAMES_PRONOSTIA],
    [grad_ewc[f]  for f in FEATURE_NAMES_PRONOSTIA],
)
print(f"Spearman EWC perm/grad : ρ = {rho_ewc:.3f}")

# HDC : permutation vs masking
hdc = data["HDC"]["global"]
perm_hdc = minmax(hdc["permutation_importance"])
mask_hdc  = minmax(hdc["feature_masking_importance"])

fig = plot_feature_importance_comparison(
    {"Permutation (HDC)": perm_hdc, "Masking (HDC)": mask_hdc},
    FEATURE_NAMES_PRONOSTIA,
    title="HDC — Permutation vs Feature Masking (normalisés) — Pronostia by_condition",
)
plt.show()

rho_hdc, _ = spearmanr(
    [perm_hdc[f] for f in FEATURE_NAMES_PRONOSTIA],
    [mask_hdc[f]  for f in FEATURE_NAMES_PRONOSTIA],
)
print(f"Spearman HDC perm/mask : ρ = {rho_hdc:.3f}")


Spearman EWC perm/grad : ρ = 0.709
Spearman HDC perm/mask : ρ = 0.543


/tmp/ipykernel_95744/16858076.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_95744/16858076.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 8 — Analyse : Pronostia by_condition

### Canal dominant : acc_horiz vs acc_vert

Le roulement PRONOSTIA supporte la charge radiale principalement sur l'axe **horizontal**.
Attendu que les features `acc_horiz` dominent. Vérifier :
- Section 4 (barplot par canal) : `acc_horiz` > `acc_vert` pour la majorité des modèles → confirme la physique
- Si `acc_vert` domine → signal inattendu, à documenter

### `temporal_position` : MCU vs laboratoire

- `temporal_position` encode la position temporelle dans la durée de vie du roulement
- **Non disponible en MCU sans horodatage externe** → `FIXME(gap1)`
- Si importance > 0.01 (Section 6) → recommandation : créer une expérience sans cette feature (voir S11-23 ablation)

### Stabilité inter-conditions

Les 3 conditions opératoires correspondent à des vitesses et charges différentes (domain shift réel).
- Heatmap uniforme → features robustes au changement de conditions → argument CL justifié
- Heatmap avec disparités fortes → domain shift fort, confirme la nécessité du CL

### Features MCU recommandées

| Feature | Justification physique | Disponibilité MCU |
|---------|------------------------|-------------------|
| `rms_acc_horiz` | Énergie de vibration sur axe de charge | ✅ |
| `kurtosis_acc_horiz` | Détection de chocs d'impact | ✅ |
| `crest_factor_acc_horiz` | Rapport crête/rms — sévérité | ✅ |
| `temporal_position` | Position dans la vie du roulement | ❌ FIXME(gap1) |
